# 10 — Analytical Model Calibration

Fit the variance_rate parameter from our boxscore data and validate the analytical win probability model.

In [1]:
import sys
sys.path.insert(0, '/Users/chriswang/Desktop/prediction-market-exploration/nba-edge')

import numpy as np
import matplotlib.pyplot as plt
from nba_edge.data.s3_reader import discover_game_ids, load_boxscores_for_game
from nba_edge.models.analytical import AnalyticalWinProb, parse_game_clock

In [ ]:
# Load all completed games and compute variance rate
all_games = []
for date in ['2026-04-21', '2026-04-22', '2026-04-23', '2026-04-24', '2026-04-25',
             '2026-04-26', '2026-04-27', '2026-04-28', '2026-04-29', '2026-04-30']:
    try:
        day_games = discover_game_ids(date)
    except Exception:
        continue
    for g in day_games:
        snaps = load_boxscores_for_game(g['game_id'], date)
        if len(snaps) < 100:  # need decent coverage
            continue
        # Check game completed (gameStatus=3)
        if snaps[-1].get('game_status', 0) != 3:
            continue
        all_games.append({
            'game_id': g['game_id'],
            'home': snaps[0]['home_team'],
            'away': snaps[0]['away_team'],
            'snapshots': snaps,
            'final_home': snaps[-1]['home_score'],
            'final_away': snaps[-1]['away_score'],
        })

print(f'Completed games loaded: {len(all_games)}')
for g in all_games:
    print(f"  {g['game_id']} {g['away']}@{g['home']} — {g['final_home']}-{g['final_away']}")

In [ ]:
# Estimate variance rate: Var(score_diff change) / time_elapsed
# Use multiple time horizons (30s, 60s, 120s, 300s) for robustness
estimates_by_horizon = {}

for horizon in [30, 60, 120, 300]:
    samples = []
    for game in all_games:
        snaps = game['snapshots']
        ts = [s['t_receipt'] for s in snaps]
        diffs = [s['home_score'] - s['away_score'] for s in snaps]
        
        for i in range(len(ts)):
            # Find snapshot approximately `horizon` seconds later
            target = ts[i] + horizon
            for j in range(i+1, len(ts)):
                if abs(ts[j] - target) < horizon * 0.1:  # within 10%
                    dt = ts[j] - ts[i]
                    ds = diffs[j] - diffs[i]
                    samples.append((ds, dt))
                    break
    
    if samples:
        ds_arr = np.array([s[0] for s in samples])
        dt_arr = np.array([s[1] for s in samples])
        # Variance rate = Var(delta_score) / mean(delta_time)
        var_rate = np.var(ds_arr) / np.mean(dt_arr)
        estimates_by_horizon[horizon] = var_rate
        print(f'  horizon={horizon:>3d}s: variance_rate = {var_rate:.4f} ({len(samples)} samples)')

# Best estimate: average across horizons
best_estimate = np.mean(list(estimates_by_horizon.values()))
print(f'\nBest estimate: {best_estimate:.4f} pts²/s')
print(f'NBA literature value: ~0.44 pts²/s')

In [ ]:
# Validate: for completed games, check model prediction at various points
# vs actual outcome
model = AnalyticalWinProb(variance_rate=best_estimate)

# Collect (prediction, outcome) pairs at 5-minute intervals
predictions = []  # (seconds_remaining, model_prob, actual_home_won)

for game in all_games:
    home_won = game['final_home'] > game['final_away']
    snaps = game['snapshots']
    
    for s in snaps:
        secs = parse_game_clock(s['period'], s['game_clock'])
        if secs <= 0:
            continue
        diff = s['home_score'] - s['away_score']
        prob = model.predict(diff, secs)
        predictions.append((secs, prob, home_won))

print(f'Total prediction points: {len(predictions):,}')

In [ ]:
# Calibration check: bin predictions, compare predicted vs actual
preds = np.array([(p[1], float(p[2])) for p in predictions])
prob_col = preds[:, 0]
outcome_col = preds[:, 1]

n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_centers = []
bin_actual = []
bin_predicted = []
bin_counts = []

for i in range(n_bins):
    mask = (prob_col >= bin_edges[i]) & (prob_col < bin_edges[i+1])
    if i == n_bins - 1:
        mask = (prob_col >= bin_edges[i]) & (prob_col <= bin_edges[i+1])
    if mask.sum() > 0:
        bin_centers.append((bin_edges[i] + bin_edges[i+1]) / 2)
        bin_predicted.append(prob_col[mask].mean())
        bin_actual.append(outcome_col[mask].mean())
        bin_counts.append(mask.sum())

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.scatter(bin_predicted, bin_actual, s=[c/50 for c in bin_counts], alpha=0.7)
for i, (x, y, c) in enumerate(zip(bin_predicted, bin_actual, bin_counts)):
    ax.annotate(f'n={c}', (x, y), textcoords='offset points', xytext=(5, 5), fontsize=8)
ax.set_xlabel('Predicted P(home_win)')
ax.set_ylabel('Actual home win rate')
ax.set_title(f'Model Calibration (variance_rate={best_estimate:.3f})')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Brier score of our model
brier = np.mean((prob_col - outcome_col) ** 2)
print(f'Model Brier score: {brier:.4f}')
print(f'(Lower is better. Naive 50/50 baseline: 0.25)')